In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [20]:
documents[0]["content"]

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

In [4]:
from minsearch import Index

index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)

index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [22]:
def search(question):
    return index.search(question, num_results=5)

# search_results = search(question, index_chunk)

In [ ]:

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("File Name: " + doc["filename"])
        lines.append("Content: " + doc["content"])
        lines.append("")

    return "\n".join(lines).strip()

# print("-" * 20 + " doc to string search results " + "-" * 20)
# print(build_context(search_results))

-------------------- doc to string search results --------------------
File Name: 01-agentic-rag/lessons/14-agentic-loop.md
Content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the ins

In [8]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)

    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

# prompt = build_prompt(question, search_results)


In [10]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text, response.usage

In [ ]:
def rag(query, model="gpt-5.4-mini"):
    context = search(query)
    prompt = build_prompt(query, context)
    response_output_text, usage_token = llm(INSTRUCTIONS, prompt, model=model)
    print(response_output_text, usage_token)

In [21]:
rag(question)

It keeps calling the model in a `while True` loop and checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is basically: **no function calls this turn = done**. ResponseUsage(input_tokens=7140, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7231)


In [1]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

NameError: name 'documents' is not defined

In [18]:
len(chunks)

295

In [19]:
index_chunk = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)

index_chunk.fit(chunks)

In [23]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):

        return self.index.search(
            query,
            num_results=num_results,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append("filename: " + doc["filename"])
            lines.append("content: " + doc["content"])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer, count = self.llm(prompt)
        return answer, count

In [24]:
assistant = RAGBase(
    index=index_chunk,
    llm_client=openai_client,
)

response_text, usage = assistant.rag("How does the agentic loop keep calling the model until it stops?")
response_text, usage

('It keeps calling the model inside a `while True` loop.\n\nEach iteration:\n- sends the current `messages` to the model,\n- checks the response for any `function_call` items,\n- runs those tools and appends the results back into `messages`.\n\nA flag, `has_function_calls`, starts as `False`.  \nIf the model calls a tool, the flag is set to `True`.\n\nAt the end of the loop:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the loop stops when the model returns a response with no function calls.',
 ResponseUsage(input_tokens=2319, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=123, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2442))

In [25]:
7140/2319

3.0789133247089264

In [26]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [27]:
def search(query: str) -> dict[str, str]:
    """
    Search the documents for entries matching the given query.
    """
    return index_chunk.search(
        query,
        num_results=5,
    )

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [29]:
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [31]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
